# 🏦 Credit Risk Analyzer — ML Model Training
## Home Credit Default Risk Dataset
- **Goal:** Train and compare Logistic Regression, Random Forest, and XGBoost models
- **Input:** data/processed/train_processed.csv, test_processed.csv
- **Evaluation Metric:** ROC-AUC, Precision, Recall, F1 Score
- **Final Model:** 

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# ML models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Evaluation metrics
from sklearn.metrics import (roc_auc_score, classification_report, 
                             confusion_matrix, roc_curve)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model saving
import joblib
import os

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [2]:
# Load preprocessed train and test data
train_df = pd.read_csv('../data/processed/train_processed.csv')
test_df = pd.read_csv('../data/processed/test_processed.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Separate features and target
X_train = train_df.drop('TARGET', axis=1)
y_train = train_df['TARGET']

X_test = test_df.drop('TARGET', axis=1)
y_test = test_df['TARGET']

print(f"\nX_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"\nTarget distribution (train):")
print(y_train.value_counts(normalize=True).mul(100).round(2))
print("✅ Data loaded successfully!")

Train shape: (246008, 233)
Test shape: (61503, 233)

X_train: (246008, 232)
X_test: (61503, 232)

Target distribution (train):
TARGET
0    91.93
1     8.07
Name: proportion, dtype: float64
✅ Data loaded successfully!


In [3]:
# Feature Scaling — only for Logistic Regression
# Tree-based models (Random Forest, XGBoost) don't need scaling

scaler = StandardScaler()

# fit_transform on train — scaler learns mean and std from train data only
X_train_scaled = scaler.fit_transform(X_train)

# transform on test — using same mean and std from train
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"Mean of first feature (should be ~0): {X_train_scaled[:, 0].mean():.4f}")
print(f"Std of first feature (should be ~1): {X_train_scaled[:, 0].std():.4f}")
print("✅ Scaling done!")

X_train_scaled shape: (246008, 232)
Mean of first feature (should be ~0): 0.0000
Std of first feature (should be ~1): 1.0000
✅ Scaling done!


In [4]:
# Model 1 — Logistic Regression (Baseline)
# max_iter=1000 — complex dataset, needs more iterations to converge
# class_weight='balanced' — handles class imbalance automatically

print("Training Logistic Regression...")

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

# Predictions
lr_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_pred = lr_model.predict(X_test_scaled)

# Evaluation
lr_auc = roc_auc_score(y_test, lr_pred_proba)

print(f"\n📊 Logistic Regression Results:")
print(f"ROC-AUC Score: {lr_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, lr_pred))
print("✅ Logistic Regression training done!")

Training Logistic Regression...

📊 Logistic Regression Results:
ROC-AUC Score: 0.7483

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.69      0.80     56538
           1       0.16      0.68      0.26      4965

    accuracy                           0.69     61503
   macro avg       0.56      0.68      0.53     61503
weighted avg       0.90      0.69      0.76     61503

✅ Logistic Regression training done!


### Model 1 — Logistic Regression (Baseline)

Trained a baseline Logistic Regression model with `class_weight='balanced'` 
to handle the 92:8 class imbalance in the dataset.

**ROC-AUC: 0.7483**

The model captures a reasonable signal given its simplicity. Recall for 
defaulters came out to 0.68 — meaning 68% of actual defaulters were correctly 
flagged. However, precision is low at 0.16, indicating a high false positive 
rate. This is expected behavior when optimizing for recall on an imbalanced dataset.

Overall accuracy of 0.69 is misleading here — ROC-AUC is the metric that matters.
This score sets our baseline. Random Forest and XGBoost should improve on this.

In [6]:
# Model 2 — Random Forest
# n_estimators=100 — 100 decision trees
# class_weight='balanced' — handles class imbalance
# n_jobs=-1 — use all CPU cores for faster training
# max_depth=10 — limit tree depth to prevent overfitting

print("Training Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Predictions
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]
rf_pred = rf_model.predict(X_test)

# Evaluation
rf_auc = roc_auc_score(y_test, rf_pred_proba)

print(f"\n📊 Random Forest Results:")
print(f"ROC-AUC Score: {rf_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, rf_pred))
print("✅ Random Forest training done!")

Training Random Forest...

📊 Random Forest Results:
ROC-AUC Score: 0.7299

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.73      0.83     56538
           1       0.16      0.60      0.26      4965

    accuracy                           0.72     61503
   macro avg       0.56      0.67      0.54     61503
weighted avg       0.89      0.72      0.78     61503

✅ Random Forest training done!


### Model 2 — Random Forest

Trained a Random Forest with 100 estimators and max_depth=10 to prevent 
overfitting. Used all CPU cores (n_jobs=-1) for faster training.

**ROC-AUC: 0.7299**

Surprisingly, Random Forest underperformed Logistic Regression here. 
Recall dropped to 0.60 — fewer defaulters caught compared to baseline. 
This is likely because max_depth=10 restricted the trees from learning 
deeper patterns, and Random Forest generally struggles more with severe 
class imbalance compared to boosting algorithms.

XGBoost should improve significantly on both metrics.

In [7]:
# Model 3 — XGBoost
# scale_pos_weight=11.4 — handles 92:8 class imbalance (282686/24825)
# learning_rate=0.05 — small steps, better generalization
# subsample=0.8 — 80% rows per tree — prevents overfitting
# colsample_bytree=0.8 — 80% features per tree — prevents overfitting

print("Training XGBoost...")

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=11.4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='auc',
    verbosity=0
)

xgb_model.fit(X_train, y_train)

# Predictions
xgb_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_pred = xgb_model.predict(X_test)

# Evaluation
xgb_auc = roc_auc_score(y_test, xgb_pred_proba)

print(f"\n📊 XGBoost Results:")
print(f"ROC-AUC Score: {xgb_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, xgb_pred))
print("✅ XGBoost training done!")

Training XGBoost...
(This will take 3-5 minutes...)

📊 XGBoost Results:
ROC-AUC Score: 0.7601

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.73      0.83     56538
           1       0.18      0.65      0.28      4965

    accuracy                           0.72     61503
   macro avg       0.57      0.69      0.55     61503
weighted avg       0.90      0.72      0.79     61503

✅ XGBoost training done!


### Model 3 — XGBoost

Trained XGBoost with 300 estimators, learning_rate=0.05, and 
scale_pos_weight=11.4 to handle class imbalance directly.

**ROC-AUC: 0.7601** — best among all three models.

Precision improved to 0.18 compared to 0.16 in both previous models. 
Recall of 0.65 is slightly lower than Logistic Regression but the overall 
ROC-AUC is highest — meaning XGBoost has the best ability to distinguish 
between defaulters and non-defaulters across all thresholds.

scale_pos_weight=11.4 gave XGBoost a clear advantage over Random Forest 
in handling the 92:8 imbalance. This confirms XGBoost as our final model.